**[PROTOTYPE]** Multi-Patch Lattice Surgery — `UnrotatedMultiPatchCoupler`

Demonstrates multi-patch Z product measurement (Z̄₁Z̄₂⋯Z̄ₙ) using the unrotated surface code.
The coupler builds a vertical quantum bus between all participating patches.

Protocol: `lightstim.qec_code.surface_code.unrotated.UnrotatedMultiPatchCoupler`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

from lightstim.qec_code.surface_code.unrotated import (
    UnrotatedSurfaceCode,
    UnrotatedSurfaceCodeExtractionBlock,
    UnrotatedMultiPatchCoupler,
)
from lightstim.ir.qec_system import QECSystem
from lightstim.ir.builder import CircuitBuilder
from lightstim.ir.tracker import SyndromeTracker
import numpy as np


# ── Helpers ──────────────────────────────────────────────────────────────────

def make_patch(d):
    """Unrotated surface code patch rotated π/2 so Z boundary faces left/right."""
    p = UnrotatedSurfaceCode(distance=d)
    p.rotate_coords(np.pi / 2)
    p.reset_rotation_angle()
    return p


def build_zz_circuit(system, coupler_name, rounds, init_basis='Z'):
    """
    Build a Z-product lattice surgery circuit.
    Sequence: init → pre-coupler SE → activate coupler → coupler SE → readout.
    init_basis='Z' (|0⟩) keeps N logicals; 'X' (|+⟩) consumes one DOF → N-1 logicals.
    """
    cp = system.coupler_patches[coupler_name]
    tracker = SyndromeTracker(
        num_qubits=system.num_qubits,
        expected_num_logicals=system.num_logicals,
    )
    builder = CircuitBuilder(tracker=tracker, system_config=system, if_detector=True)
    builder.write_coordinates()

    non_coupler = {q: init_basis for q in system.data_indices
                   if system.index_to_owner_map[q] != coupler_name}
    builder.initialize(init_dict=non_coupler, n=system.num_qubits)

    se = UnrotatedSurfaceCodeExtractionBlock(system)
    builder.apply_syndrome_extraction(circuit_chunk=se.circuit, rounds=rounds)

    builder.activate_coupler(coupler_name)
    coupler_data = {
        system.local_to_global_map[coupler_name][q]: 'X'
        for q in cp.data_indices
    }
    builder.initialize(init_dict=coupler_data, n=system.num_qubits)

    se2 = UnrotatedSurfaceCodeExtractionBlock(system)
    builder.apply_syndrome_extraction(circuit_chunk=se2.circuit, rounds=rounds)

    builder.apply_data_readout(final_measurements={**non_coupler, **coupler_data})
    return builder.circuit

## Coupler geometry (no detectors)

Qubit layout and stabilizer structure for different patch configurations.
For pure geometry inspection, patches don't need rotation.

In [ ]:
# 2-patch: side-by-side, vertical corridor at x=7
system = QECSystem()
system.add_patch(UnrotatedSurfaceCode(distance=3), name='p1')
system.add_patch(UnrotatedSurfaceCode(distance=3), name='p2', offset=(10, 0))
system.register_coupler(UnrotatedMultiPatchCoupler(), ['p1', 'p2'], 'c',
                         path_axis='vertical', center_axis=7.0)
cp = system.coupler_patches['c']
print(f"2-patch: {len(cp.data_indices)} data, {len(cp.stabilizers)} stabilizers")

system.activate_coupler('c')
tracker = SyndromeTracker(num_qubits=system.num_qubits, expected_num_logicals=system.num_logicals)
builder = CircuitBuilder(tracker=tracker, system_config=system, if_detector=False)
builder.write_coordinates()
builder.initialize({q: 'Z' for q in system.data_indices}, n=system.num_qubits)
builder.apply_syndrome_extraction(
    circuit_chunk=UnrotatedSurfaceCodeExtractionBlock(system).circuit, rounds=1)
system.deactivate_coupler('c')
# builder.circuit.diagram("detslice-with-ops-svg")  # comment out before commit

### 3-patch ZZZ

In [ ]:
# 3-patch: p1 (left), p2 (right), p3 (left-bottom)
system = QECSystem()
system.add_patch(UnrotatedSurfaceCode(distance=3), name='p1')
system.add_patch(UnrotatedSurfaceCode(distance=3), name='p2', offset=(10, 0))
system.add_patch(UnrotatedSurfaceCode(distance=3), name='p3', offset=(0, 6))
system.register_coupler(UnrotatedMultiPatchCoupler(), ['p1', 'p2', 'p3'], 'zzz',
                         path_axis='vertical', center_axis=7.0)
cp = system.coupler_patches['zzz']
print(f"3-patch: {len(cp.data_indices)} data, {len(cp.stabilizers)} stabilizers")

system.activate_coupler('zzz')
tracker = SyndromeTracker(num_qubits=system.num_qubits, expected_num_logicals=system.num_logicals)
builder = CircuitBuilder(tracker=tracker, system_config=system, if_detector=False)
builder.write_coordinates()
builder.initialize({q: 'Z' for q in system.data_indices}, n=system.num_qubits)
builder.apply_syndrome_extraction(
    circuit_chunk=UnrotatedSurfaceCodeExtractionBlock(system).circuit, rounds=1)
system.deactivate_coupler('zzz')
# builder.circuit.diagram("detslice-with-ops-svg")  # comment out before commit

### Endpoint patch geometry

5-patch case: 4 side patches + 1 endpoint. `start_patch=4` marks the endpoint patch,
which connects to the bottom of the vertical corridor rather than from the side.

In [ ]:
# 5-patch: 4 side patches + 1 endpoint at bottom (start_patch=4)
system = QECSystem()
for name, off in [('p1',(-2,2)), ('p2',(10,2)), ('p3',(-2,8)), ('p4',(10,8)), ('p_end',(4,18))]:
    system.add_patch(UnrotatedSurfaceCode(distance=3), name=name, offset=off)
system.register_coupler(
    UnrotatedMultiPatchCoupler(), ['p1', 'p2', 'p3', 'p4', 'p_end'], 'ex',
    path_axis='vertical', start_patch=4,
)
cp = system.coupler_patches['ex']
print(f"5-patch (1 endpoint): {len(cp.data_indices)} data, {len(cp.stabilizers)} stabilizers")

system.activate_coupler('ex')
tracker = SyndromeTracker(num_qubits=system.num_qubits, expected_num_logicals=system.num_logicals)
builder = CircuitBuilder(tracker=tracker, system_config=system, if_detector=False)
builder.write_coordinates()
builder.initialize({q: 'Z' for q in system.data_indices}, n=system.num_qubits)
builder.apply_syndrome_extraction(
    circuit_chunk=UnrotatedSurfaceCodeExtractionBlock(system).circuit, rounds=1)
system.deactivate_coupler('ex')
# builder.circuit.diagram("detslice-with-ops-svg")  # comment out before commit

---

## Lattice Surgery Experiments

All patches rotated π/2 so their Z boundaries face the vertical corridor.
`build_zz_circuit()` handles: init → pre-coupler SE → activate → coupler SE → readout.

### Exp 1: 2-patch ZZ measurement

In [ ]:
d = 3
system = QECSystem()
system.add_patch(make_patch(d), name='p1')
system.add_patch(make_patch(d), name='p2', offset=(10, 0))
system.register_coupler(UnrotatedMultiPatchCoupler(), ['p1', 'p2'], 'c12',
                         path_axis='vertical', center_axis=7.0)

circuit = build_zz_circuit(system, 'c12', rounds=d)
print(f"d={d}: {circuit.num_qubits} qubits, {circuit.num_detectors} det, {circuit.num_observables} obs")
dem = circuit.detector_error_model(decompose_errors=True)
print(f"DEM OK: {dem.num_detectors} det, {dem.num_observables} obs")
# circuit.diagram("detslice-with-ops-svg")  # comment out before commit

### Exp 2: 3-patch ZZZ measurement

In [ ]:
d = 3
system = QECSystem()
system.add_patch(make_patch(d), name='p1')
system.add_patch(make_patch(d), name='p2', offset=(10, 0))
system.add_patch(make_patch(d), name='p3', offset=(0, 6))
system.register_coupler(UnrotatedMultiPatchCoupler(), ['p1', 'p2', 'p3'], 'zzz',
                         path_axis='vertical', center_axis=7.0)

circuit = build_zz_circuit(system, 'zzz', rounds=d)
print(f"d={d}: {circuit.num_qubits} qubits, {circuit.num_detectors} det, {circuit.num_observables} obs")
dem = circuit.detector_error_model(decompose_errors=True)
print(f"DEM OK: {dem.num_detectors} det, {dem.num_observables} obs")
# circuit.diagram("detslice-with-ops-svg")  # comment out before commit

### Exp 3: 5-patch system, selective 4-patch ZZZZ

5 patches total; only 4 participate. `p1` (top-left) is idle — selective joint
measurement as used in distillation circuits.

In [ ]:
d = 3
patch_layout = [
    ('p1', (-2, 2)),   # left-top  — IDLE
    ('p2', (10, 2)),   # right-top
    ('p3', (-2, 8)),   # left-mid
    ('p4', (10, 8)),   # right-mid
    ('p5', (-2, 14)),  # left-bottom
]
system = QECSystem()
for name, off in patch_layout:
    system.add_patch(make_patch(d), name=name, offset=off)

# Register coupler for p2-p5 only (p1 is idle)
system.register_coupler(UnrotatedMultiPatchCoupler(), ['p2', 'p3', 'p4', 'p5'], 'z4',
                         path_axis='vertical', center_axis=6.0)

circuit = build_zz_circuit(system, 'z4', rounds=d)
print(f"d={d}: {circuit.num_qubits} qubits, {circuit.num_detectors} det, {circuit.num_observables} obs")
dem = circuit.detector_error_model(decompose_errors=True)
print(f"DEM OK: {dem.num_detectors} det, {dem.num_observables} obs")
# circuit.diagram("detslice-with-ops-svg")  # comment out before commit

### Exp 4: Mixed code distances (d=3 and d=4)

5 patches; `p3` is idle; `p1,p2,p4,p5` participate in ZZZZ.

> **Note**: All interacting patches must be side patches — perpendicular endpoint
> boundaries cause CNOT scheduling conflicts. See `skills/gotchas/SKILL.md` §8.

In [ ]:
d = 3
patch_layout = [
    ('p1', 3, (-2, 2)),   # left-top
    ('p2', 3, (-2, 8)),   # left-mid
    ('p3', 3, (12, 2)),   # right-top — IDLE (shifted to avoid corridor)
    ('p4', 4, (10, 8)),   # right-mid, d=4
    ('p5', 3, (-2, 16)),  # left-bottom
]
system = QECSystem()
for name, dist, off in patch_layout:
    system.add_patch(make_patch(dist), name=name, offset=off)

# p1,p2,p4,p5 interact; p3 is idle
system.register_coupler(UnrotatedMultiPatchCoupler(), ['p1', 'p2', 'p4', 'p5'], 'z4',
                         path_axis='vertical', center_axis=6.0)

circuit = build_zz_circuit(system, 'z4', rounds=d)
print(f"d={d}: {circuit.num_qubits} qubits, {circuit.num_detectors} det, {circuit.num_observables} obs")
dem = circuit.detector_error_model(decompose_errors=True)
print(f"DEM OK: {dem.num_detectors} det, {dem.num_observables} obs")
# circuit.diagram("detslice-with-ops-svg", tick=range(30, 50))  # comment out before commit

---

## Observable count: init basis vs Z measurement

For an N-patch Z product measurement:
- Init `|0⟩` (Z commutes): N logical observables preserved
- Init `|+⟩` (X anti-commutes): N−1 observables (one DOF consumed by the measurement)

In [ ]:
def _make_n_system(n_patches, d=3):
    n_left  = (n_patches + 1) // 2
    n_right = n_patches // 2
    system = QECSystem()
    names = []
    for i in range(n_left):
        system.add_patch(make_patch(d), name=f'L{i}', offset=(-2, i * 6))
        names.append(f'L{i}')
    for i in range(n_right):
        system.add_patch(make_patch(d), name=f'R{i}', offset=(10, i * 6))
        names.append(f'R{i}')
    system.register_coupler(UnrotatedMultiPatchCoupler(), names, 'c',
                             path_axis='vertical', center_axis=6.0)
    return system


print(f"{'N':>3}  {'basis':>6}  {'obs':>5}  {'expected':>9}  {'ok':>4}")
for n in [2, 3, 4]:
    for basis, expected in [('Z', n), ('X', n - 1)]:
        system = _make_n_system(n)
        circuit = build_zz_circuit(system, 'c', rounds=3, init_basis=basis)
        obs = circuit.num_observables
        label = '|0⟩' if basis == 'Z' else '|+⟩'
        print(f"{n:>3}  {label:>6}  {obs:>5}  {expected:>9}  {'✓' if obs == expected else '✗':>4}")